# Bird Species Audio Classification (BirdCLEF+ 2025) — Google Colab GPU Pipeline

This notebook is **fully self-contained** and runs end-to-end training, evaluation, Grad-CAM visualization, and Gradio deployment on a **Google Colab GPU** (`T4` / `L4` / `A100`).

It supports **two dataset modes**:
- **Mode A — iNaturalist Real Bird Dataset (Zero Setup):** Automatically downloads research-grade bird recordings via the public iNaturalist API (no Kaggle key required).
- **Mode B — Full Kaggle BirdCLEF+ 2025 Dataset (206 Species):** Downloads the full competition dataset using your `kaggle.json` credentials.

## 1. Check GPU & Install System Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected! For fast training, go to Runtime -> Change runtime type -> T4 GPU.")
!nvidia-smi || true
!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

## 2. Clone Repository & Install Python Requirements

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Haricharanv/bird-audio-classification.git"
REPO_DIR = Path("/content/bird-audio-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
env = {**os.environ, "HF_HUB_OFFLINE": "1"}
subprocess.run(["pytest", "tests/", "-v"], env=env, check=True)

## 3. Optional: Mount Google Drive (Recommended to Persist Checkpoints)
Set `USE_GOOGLE_DRIVE = True` to save trained `best_model.pt` bundles directly to your Google Drive.

In [ ]:
USE_GOOGLE_DRIVE = False  # Set True to save checkpoints/models to Google Drive
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/bird_audio_runs")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Checkpoints will be copied to: {DRIVE_SAVE_DIR}")

## 4. Choose Dataset Mode (`"inat"` or `"kaggle"`)

- **`"inat"`**: Downloads 40 research-grade recordings per species for 8 iconic bird species via iNaturalist API (no auth needed, ~2-3 min).
- **`"kaggle"`**: Downloads the full **BirdCLEF+ 2025** competition dataset (upload `kaggle.json` when prompted; make sure you accepted the competition rules on Kaggle).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import os
from pathlib import Path
import subprocess
import tempfile
import time
import urllib.parse
import urllib.request
import numpy as np
import pandas as pd
import soundfile as sf

DATASET_MODE = "inat"  # "inat" | "kaggle"
INAT_PER_SPECIES = 40

INAT_SPECIES = [
    ("common_blackbird", "Common Blackbird", "Turdus merula"),
    ("northern_cardinal", "Northern Cardinal", "Cardinalis cardinalis"),
    ("american_crow", "American Crow", "Corvus brachyrhynchos"),
    ("common_cuckoo", "Common Cuckoo", "Cuculus canorus"),
    ("house_sparrow", "House Sparrow", "Passer domesticus"),
    ("mourning_dove", "Mourning Dove", "Zenaida macroura"),
    ("great_tit", "Great Tit", "Parus major"),
    ("american_robin", "American Robin", "Turdus migratorius"),
]
UA = "bird-audio-classification-edu/1.0"

def _fetch_inat_obs(sci_name, page=1, per_page=80):
    params = urllib.parse.urlencode({
        "taxon_name": sci_name,
        "sounds": "true",
        "quality_grade": "research",
        "per_page": per_page,
        "page": page,
        "order": "desc",
        "order_by": "votes",
    })
    req = urllib.request.Request(f"https://api.inaturalist.org/v1/observations?{params}", headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=20) as resp:
        return json.loads(resp.read().decode("utf-8")).get("results", [])

def _download_clip(url, dest_ogg, max_sec=20, min_sec=2.5, min_rms=0.005, max_bytes=12*1024*1024):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": UA})
        with urllib.request.urlopen(req, timeout=15) as resp:
            cl = resp.headers.get("Content-Length")
            if cl and int(cl) > max_bytes:
                return False
            raw = resp.read(max_bytes + 1)
            if len(raw) > max_bytes or len(raw) < 2048:
                return False
    except Exception:
        return False

    suffix = Path(urllib.parse.urlparse(url).path).suffix or ".audio"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=True) as tmp:
        tmp.write(raw)
        tmp.flush()
        dest_ogg.parent.mkdir(parents=True, exist_ok=True)
        proc = subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error", "-i", tmp.name,
            "-t", str(max_sec), "-ac", "1", "-ar", "32000", "-c:a", "libvorbis", "-q:a", "4", str(dest_ogg)
        ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=30)
        if proc.returncode != 0 or not dest_ogg.is_file():
            dest_ogg.unlink(missing_ok=True)
            return False
    try:
        audio, sr = sf.read(str(dest_ogg), dtype="float32")
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        rms = float(np.sqrt(np.mean(np.square(audio)))) if len(audio) > 0 else 0.0
        if (len(audio) / float(sr)) < min_sec or rms < min_rms or np.isnan(audio).any():
            dest_ogg.unlink(missing_ok=True)
            return False
        return True
    except Exception:
        dest_ogg.unlink(missing_ok=True)
        return False

if DATASET_MODE == "inat":
    RAW_DIR = Path("/content/data/real_birds")
    CACHE_DIR = Path("/content/cache_real")
    META_DIR = Path("data/metadata_real")
    NOISE_PATH = Path("data/noise/noise_bank_real.npy")
    audio_root = RAW_DIR / "train_audio"
    audio_root.mkdir(parents=True, exist_ok=True)

    all_rows, tax_rows = [], []
    for label, common, sci in INAT_SPECIES:
        candidates, seen_authors, seen_obs = [], set(), set()
        for page in (1, 2, 3):
            results = _fetch_inat_obs(sci, page=page, per_page=80)
            if not results:
                break
            for obs in results:
                oid = str(obs.get("id", ""))
                author = str((obs.get("user") or {}).get("login") or f"inat_{oid}").strip()
                sounds = obs.get("sounds") or []
                s_url = next((s.get("file_url") for s in sounds if s.get("file_url")), None)
                if not s_url or oid in seen_obs or author in seen_authors:
                    continue
                seen_obs.add(oid); seen_authors.add(author)
                candidates.append((oid, author, s_url))
                if len(candidates) >= INAT_PER_SPECIES * 2:
                    break
            if len(candidates) >= INAT_PER_SPECIES * 2:
                break
            time.sleep(0.2)

        rows, pending = [], []
        for oid, author, s_url in candidates:
            rel = f"{label}/{label}_{oid}.ogg"
            out_p = audio_root / rel
            if out_p.is_file() and out_p.stat().st_size > 1000:
                rows.append({"primary_label": label, "secondary_labels": "[]", "filename": rel, "author": author, "collection": "iNat", "rating": ""})
            else:
                pending.append((oid, author, s_url, rel, out_p))

        cursor = 0
        with ThreadPoolExecutor(max_workers=8) as pool:
            while len(rows) < INAT_PER_SPECIES and cursor < len(pending):
                batch = pending[cursor : cursor + (INAT_PER_SPECIES - len(rows))]
                cursor += len(batch)
                fmap = {pool.submit(_download_clip, u, p): (a, r, p) for _, a, u, r, p in batch}
                for fut in as_completed(fmap):
                    a, r, p = fmap[fut]
                    if fut.result() and len(rows) < INAT_PER_SPECIES:
                        rows.append({"primary_label": label, "secondary_labels": "[]", "filename": r, "author": a, "collection": "iNat", "rating": ""})
                    elif len(rows) >= INAT_PER_SPECIES:
                        p.unlink(missing_ok=True)
        print(f"[{label}] ({common} / {sci}): collected {len(rows)} recordings.")
        all_rows.extend(rows)
        tax_rows.append({"primary_label": label, "common_name": common, "scientific_name": sci, "class_name": "Aves"})

    pd.DataFrame(all_rows).to_csv(RAW_DIR / "train.csv", index=False)
    pd.DataFrame(tax_rows).to_csv(RAW_DIR / "taxonomy.csv", index=False)
    print(f"Saved {len(all_rows)} recordings across {len(tax_rows)} species to {RAW_DIR}")

elif DATASET_MODE == "kaggle":
    from google.colab import files
    RAW_DIR = Path("/content/data/birdclef-2025")
    CACHE_DIR = Path("/content/cache_birdclef")
    META_DIR = Path("data/metadata_birdclef")
    NOISE_PATH = Path("data/noise/noise_bank_birdclef.npy")
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    if not (kaggle_dir / "kaggle.json").is_file():
        print("Upload your kaggle.json token file:")
        uploaded = files.upload()
        for name, data in uploaded.items():
            (kaggle_dir / "kaggle.json").write_bytes(data)
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "competitions", "download", "-c", "birdclef-2025", "-p", str(RAW_DIR)], check=True)
    subprocess.run(["unzip", "-q", "-o", str(RAW_DIR / "birdclef-2025.zip"), "-d", str(RAW_DIR)], check=True)
else:
    raise ValueError("DATASET_MODE must be 'inat' or 'kaggle'")

## 5. Prepare Splits, Noise Bank, and Cached Energy Chunks

In [ ]:
BASE_CFG = "configs/effnet_b0.yaml"

subprocess.run([
    "python", "scripts/prepare_data.py",
    "--config", BASE_CFG,
    "debug=false",
    f"data.raw_dir={RAW_DIR}",
    f"data.cache_dir={CACHE_DIR}",
    f"data.metadata_dir={META_DIR}",
    f"data.noise_bank_path={NOISE_PATH}",
    "data.min_files_per_species=10",
    "data.max_chunks_per_species=200",
    "data.split_by=filename",
], check=True)

## 6. Train Model on Colab GPU (`EfficientNet-B0`, `CRNN`, or `AST`)

Pick any config (`configs/effnet_b0.yaml`, `configs/crnn.yaml`, `configs/ast.yaml`, or `configs/baseline_cnn.yaml`). GPU mixed precision (`fp16`) is enabled automatically.

In [ ]:
TRAIN_CFG = "configs/effnet_b0.yaml"  # or configs/crnn.yaml, configs/ast.yaml
EPOCHS = 25
BATCH_SIZE = 32

subprocess.run([
    "python", "scripts/train.py",
    "--config", TRAIN_CFG,
    "debug=false",
    "system.device=cuda",
    "system.mixed_precision=fp16",
    "system.num_workers=2",
    "system.pin_memory=true",
    f"data.raw_dir={RAW_DIR}",
    f"data.cache_dir={CACHE_DIR}",
    f"data.metadata_dir={META_DIR}",
    f"data.noise_bank_path={NOISE_PATH}",
    f"train.epochs={EPOCHS}",
    f"train.batch_size={BATCH_SIZE}",
], check=True)

## 7. Evaluate Best Checkpoint on Held-Out Test Split & View Plots

In [ ]:
import glob
from IPython.display import Image, display

checkpoints = sorted(glob.glob("runs/*/best_model.pt"), key=os.path.getmtime, reverse=True)
assert checkpoints, "No checkpoint found in runs/!"
BEST_CKPT = Path(checkpoints[0]).resolve()
print(f"Evaluating latest checkpoint: {BEST_CKPT}")

subprocess.run([
    "python", "scripts/evaluate.py",
    "--checkpoint", str(BEST_CKPT),
    "--config", TRAIN_CFG,
    "--split", "test",
    f"data.raw_dir={RAW_DIR}",
    f"data.cache_dir={CACHE_DIR}",
    f"data.metadata_dir={META_DIR}",
], check=True)

eval_dir = BEST_CKPT.parent / "evaluation_test"
for img_name in ["confusion_matrix.png", "sample_gradcam.png"]:
    img_path = eval_dir / img_name
    if img_path.is_file():
        display(Image(filename=str(img_path)))

## 8. Export Trained Model Checkpoint for Local Use
Copies `best_model.pt` and `taxonomy.csv` into a `.zip` bundle (saved to Google Drive or downloaded to your machine) so you can unzip it into `bird-audio-classification/runs/` locally.

In [ ]:
import shutil

run_dir = BEST_CKPT.parent
tax_src = RAW_DIR / "taxonomy.csv"
if tax_src.is_file():
    shutil.copy2(tax_src, run_dir / "taxonomy.csv")

bundle_path = Path("/content") / f"{run_dir.name}_bundle.zip"
shutil.make_archive(str(bundle_path.with_suffix("")), "zip", root_dir=run_dir.parent, base_dir=run_dir.name)
print(f"Created export archive: {bundle_path}")

if USE_GOOGLE_DRIVE:
    shutil.copy2(bundle_path, DRIVE_SAVE_DIR / bundle_path.name)
    print(f"Copied archive to Google Drive: {DRIVE_SAVE_DIR / bundle_path.name}")
else:
    from google.colab import files
    files.download(str(bundle_path))

## 9. Launch Interactive Gradio Web Demo (Public Share URL)

In [ ]:
import gradio as gr
os.environ["CHECKPOINT_PATH"] = str(BEST_CKPT)

import app.app as bird_app
bird_app.MODEL_CACHE.clear()

# Populate Common Name (Scientific Name) labels if taxonomy.csv exists
tax_map = {}
tax_file = BEST_CKPT.parent / "taxonomy.csv"
if not tax_file.is_file():
    tax_file = RAW_DIR / "taxonomy.csv"
if tax_file.is_file():
    tdf = pd.read_csv(tax_file, dtype=str)
    if "primary_label" in tdf.columns and "common_name" in tdf.columns:
        for _, r in tdf.iterrows():
            lbl = str(r["primary_label"]).strip()
            com = str(r.get("common_name", "")).strip()
            sci = str(r.get("scientific_name", "")).strip()
            if com and com.lower() != "nan":
                tax_map[lbl] = f"{com} ({sci})" if sci and sci.lower() != "nan" else com

if not hasattr(bird_app, "load_species_display_names") and tax_map:
    _orig_predict = bird_app.predict_bird_audio
    def _predict_with_names(audio_filepath):
        top3, fig_spec, fig_tl = _orig_predict(audio_filepath)
        return {tax_map.get(k, k): v for k, v in top3.items()}, fig_spec, fig_tl
    bird_app.predict_bird_audio = _predict_with_names

# Copy one held-out test recording per species into app/examples for interactive testing
splits_df = pd.read_csv(META_DIR / "splits.csv")
test_df = splits_df[splits_df["split"] == "test"]
ex_dir = Path("app/examples")
ex_dir.mkdir(parents=True, exist_ok=True)
copied_examples = []
for sp, grp in test_df.groupby("primary_label"):
    src_clip = RAW_DIR / "train_audio" / grp.iloc[0]["filename"]
    if src_clip.is_file():
        dst_clip = ex_dir / f"{sp}_test.ogg"
        shutil.copy2(src_clip, dst_clip)
        copied_examples.append([str(dst_clip)])

if not hasattr(bird_app, "load_species_display_names") and copied_examples:
    bird_app.create_demo_audio_examples = lambda _dir: copied_examples

demo = bird_app.build_app()
demo.launch(share=True, debug=False)